In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map.add_basemap('HYBRID')
map.default_style = {'cursor': 'crosshair'}


map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [3]:
start_date = '2024-01-01'
end_date = '2024-06-01'


In [4]:
dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
    .filterBounds(region) \
    .filterDate(start_date, end_date) \
    .mode() \
    .clip(region)

In [ ]:
classification_vis = {
    'min': 0,
    'max': 8,
    'palette': [
        '419BDF', # 0: Water (Blue) Permanent and seasonal water bodies
        '397D49', # 1: Trees (Dark Green) Includes primary and secondary forests, as well as large-scale plantations
        '88B053', # 2: Grass (Light Green) 	Natural grasslands, livestock pastures, and parks
        '7A87C6', # 3: Flooded Vegetation (Purple) Mangroves and other inundated ecosystems
        'E49635', # 4: Crops (Orange) Include row crops and paddy crops
        'DFC35A', # 5: Shrub and Scrub (Yellow) Sparse to dense open vegetation consisting of shrubs
        'C4281B', # 6: Built Area (Red) Low- and high-density buildings, roads, and urban open space
        'A59B8F', # 7: Bare Ground (Grey) Deserts and exposed rock
        'B39FE1'  # 8: Snow or Ice (White)	Permanent and seasonal snow cover
    ]
    #these classes are from official documentation https://developers.google.com/earth-engine/tutorials/community/introduction-to-dynamic-world-pt-1
}

In [7]:
stats = dw.select('label').reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=region,
    scale=10,
    maxPixels=1e9
)

counts_dict = stats.get('label').getInfo()
total_pixels = sum(counts_dict.values())
class_names = [
    'Water', 'Trees', 'Grass', 'Flooded Vegetation', 
    'Crops', 'Shrub and Scrub', 'Built Area', 
    'Bare Ground', 'Snow or Ice'
]

for class_index_str, pixel_count in counts_dict.items():
    class_index = int(class_index_str)
    percentage = (pixel_count / total_pixels) * 100
    
    # Get the name (if index is within 0-8)
    name = class_names[class_index] if 0 <= class_index < 9 else f"class {class_index}"
    
    print(f"{name}: {percentage:.2f}%")

Water: 7.85%
Trees: 5.33%
Grass: 0.35%
Flooded Vegetation: 0.32%
Crops: 10.43%
Shrub and Scrub: 1.92%
Built Area: 71.22%
Bare Ground: 2.57%
Snow or Ice: 0.01%


In [10]:
map.addLayer(dw.select('label'), classification_vis, 'google dynamic world results')
map

Map(bottom=226772.0, center=[23.78031811836225, 90.43430328369142], controls=(WidgetControl(options=['position…